# CLIP Text-Fusion Training — Branch B

**Branch:** `feature/clip-text-fusion` (Option B from the FYP CLIP integration plan)

**Claim.** Directly correlate the patient's symptom description with the image at prediction
time by fusing MobileNetV3's own visual features with CLIP's text embedding of the symptom
text, through a small trained fusion head. Unlike branches A and C, CLIP's text tower runs
*live* at inference — this is the branch that most literally answers the panel's "image and
symptom description are processed separately" feedback.

**Alternatives seriously considered.**
- *Decision-level fusion* (combine MobileNetV3's 7-class probabilities with a CLIP zero-shot
  text-similarity score, similar in spirit to how branch A's teachers are combined) — rejected
  for this branch specifically because it would look too similar to what A already does, and
  loses information: MobileNetV3's already-compressed 7-way probability distribution is a much
  weaker fusion input than its raw 512-dim penultimate feature vector.
- *Multilingual CLIP text tower* (as considered for the translation-vs-multilingual-CLIP
  question) — rejected in favor of a translation pipeline in front of the same English-only
  `ViT-B-32` (openai) CLIP used in A and C, keeping the CLIP model itself identical across all
  three branches; only the new translation/transliteration stages differ.

**Rejection criteria.** If the fused model does not outperform MobileNetV3 alone (image-only)
on the same test set, or if a sanity check with deliberately mismatched text doesn't hurt
accuracy (see the "Does the model actually use the text?" section below), the fusion mechanism
is not adding real signal and should be reported as a negative result, not shipped.

**The multilingual pipeline this notebook's fusion head must be compatible with** (implemented
in `inference/server.py`'s `/predict_fused`, not here): English text goes straight to CLIP;
Urdu-script text is translated to English first via `facebook/nllb-200-distilled-600M`; Roman
Urdu is transliterated to Urdu script first via `Mavkif/m2m100_rup_rur_to_ur`, then translated
the same way. All three paths converge to English before CLIP ever embeds the text — verified
directly against real symptom phrases before this pipeline was chosen (not assumed): the
originally-considered `Helsinki-NLP/opus-mt-ur-en` mistranslated exactly the clinical
vocabulary that matters ("itching" → "signing", "redness" → "black"), while NLLB got all of
the same test phrases correct, hence the swap.

**Why this notebook trains on English-only synthetic text, and why that's not a shortcut here**:
since every language is translated to English before CLIP sees it (both at training time in
this notebook and at inference time in the server), there is no cross-lingual-transfer
assumption being made — training and inference operate in the same English-only embedding
space by construction. No real (image, symptom-text, label) dataset exists anywhere in this
repo, so the text side of every training pair here is a synthetic, patient-phrased template
per class, not real patient language — a real, stated limitation (see summary at the end),
not a hidden one.

**Assumes** `03_model_training.ipynb` has already been run, producing
`../models/student_large_distilled.pth`.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.transforms as transforms
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import models
import pandas as pd
import numpy as np
import random
from PIL import Image
from sklearn.metrics import classification_report, accuracy_score
import open_clip

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

In [ ]:
# ── Config ────────────────────────────────────────────────────────────
CLASS_NAMES = [
    "Vitiligo", "Melasma", "Psoriasis", "Eczema",
    "Tinea", "Contact Dermatitis", "Seborrheic Dermatitis",
]  # order must match the numeric_label encoding in data/processed/*.csv
NUM_CLASSES = len(CLASS_NAMES)

IMG_SIZE = 224
CLIP_MODEL_NAME = "ViT-B-32-quickgelu"  # matches A/C — "-quickgelu" matches the "openai" weights' activation
CLIP_PRETRAINED = "openai"

FUSION_HIDDEN_DIM = 256
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 1e-4
NUM_EPOCHS = 40

BASELINE_STUDENT_CHECKPOINT = "../models/student_large_distilled.pth"
FUSION_HEAD_CHECKPOINT = "../models/fusion_head.pth"

TRAIN_CSV = "../data/processed/train.csv"
VAL_CSV = "../data/processed/val.csv"
TEST_CSV = "../data/processed/test.csv"

## Synthetic proxy symptom text

Patient-phrased, first-person descriptions per class — matching the tone of the real example
transcripts already in `server/src/utils/mockData.js` (e.g. *"I have itching on my arms for
two weeks"*), not clinical/dermatology-textbook phrasing like the CLIP zero-shot prompts used
in branches A and C. This is a deliberate difference: those prompts are written for CLIP's own
"a photo of X" zero-shot convention; these are written to look like what `/predict_fused` will
actually receive from a real (translated) patient message.

In [ ]:
SYMPTOM_TEXT_TEMPLATES = {
    "Vitiligo": [
        "I have white patches on my skin that keep spreading.",
        "There are pale, depigmented spots on my hands and face.",
        "My skin has lost its color in some areas.",
    ],
    "Melasma": [
        "I have brown patches on my face that won't go away.",
        "Dark blotchy spots have appeared on my cheeks and forehead.",
        "My face has uneven, darker patches of skin.",
    ],
    "Psoriasis": [
        "I have thick, scaly red patches on my elbows and knees.",
        "My skin is flaky and covered in silvery scales.",
        "There are raised, itchy red plaques on my skin.",
    ],
    "Eczema": [
        "I have itching on my arms for two weeks.",
        "My skin is dry, cracked, and very itchy.",
        "There's red, inflamed skin that itches constantly.",
    ],
    "Tinea": [
        "I have a red, ring-shaped rash that's spreading.",
        "There's a circular, itchy patch with a clear center on my skin.",
        "My skin has a scaly ring-shaped mark.",
    ],
    "Contact Dermatitis": [
        "Red patches appeared on my face with burning.",
        "My skin got red and swollen after touching something.",
        "I have an itchy, irritated rash after using a new soap.",
    ],
    "Seborrheic Dermatitis": [
        "I have greasy, yellowish scales on my scalp and face.",
        "My skin is flaky and oily in patches.",
        "There's dry, flaky skin with yellowish scales since last month.",
    ],
}
assert list(SYMPTOM_TEXT_TEMPLATES.keys()) == CLASS_NAMES, "template keys must match CLASS_NAMES order"
print(f"{sum(len(v) for v in SYMPTOM_TEXT_TEMPLATES.values())} synthetic symptom-text templates across {NUM_CLASSES} classes")

## Loading the frozen MobileNetV3-Large student
Already trained by `03_model_training.ipynb` — never updated here. Only its 512-dim penultimate feature (before the final classification `Linear`) is used, not its 7-class output.

In [ ]:
def build_student_large(num_classes=7):
    model = models.mobilenet_v3_large(weights=None)
    in_features = model.classifier[0].in_features
    model.classifier = nn.Sequential(
        nn.Linear(in_features, 512),
        nn.Hardswish(),
        nn.Dropout(p=0.3),
        nn.Linear(512, num_classes),
    )
    return model

student = build_student_large(NUM_CLASSES)
student.load_state_dict(torch.load(BASELINE_STUDENT_CHECKPOINT, map_location=device, weights_only=True))
student = student.to(device)
student.eval()
for param in student.parameters():
    param.requires_grad = False

print(f"MobileNetV3-Large student loaded from {BASELINE_STUDENT_CHECKPOINT}, frozen ({sum(p.numel() for p in student.parameters()):,} params)")


@torch.no_grad()
def extract_mobilenet_features(image_batch):
    """Runs the student up through its 512-dim penultimate layer (classifier[0] + Hardswish),
    stopping before Dropout + the final 7-class Linear. This is the fusion head's image input."""
    x = student.features(image_batch)
    x = student.avgpool(x)
    x = torch.flatten(x, 1)
    x = student.classifier[0](x)   # Linear(in_features, 512)
    x = student.classifier[1](x)   # Hardswish
    return x  # [batch, 512]

## Loading frozen CLIP (text tower only used here, but the full paired model loads)
Identical model/weights to branches A and C — `ViT-B-32-quickgelu` (openai). Only `encode_text` is ever called in this notebook or in `/predict_fused`; the vision tower loads but is unused, exactly as planned ("CLIP's text tower, not the vision tower, runs live").

In [ ]:
print(f"Loading CLIP {CLIP_MODEL_NAME} ({CLIP_PRETRAINED})...")
clip_model, _, _ = open_clip.create_model_and_transforms(CLIP_MODEL_NAME, pretrained=CLIP_PRETRAINED)
clip_tokenizer = open_clip.get_tokenizer(CLIP_MODEL_NAME)
clip_model = clip_model.to(device)
clip_model.eval()
for param in clip_model.parameters():
    param.requires_grad = False

CLIP_EMBED_DIM = clip_model.text_projection.shape[1] if hasattr(clip_model, "text_projection") else 512
print(f"CLIP loaded and frozen ({sum(p.numel() for p in clip_model.parameters()):,} params), text embed dim: {CLIP_EMBED_DIM}")


@torch.no_grad()
def embed_symptom_text(texts):
    """texts: list[str]. Returns L2-normalized CLIP text embeddings, [batch, CLIP_EMBED_DIM]."""
    tokens = clip_tokenizer(texts).to(device)
    embeddings = clip_model.encode_text(tokens)
    embeddings = embeddings / embeddings.norm(dim=-1, keepdim=True)
    return embeddings

## The fusion head — the "small learned layer" from the plan

Concatenates MobileNetV3's 512-dim feature with CLIP's 512-dim text embedding (1024 total),
through one hidden layer, down to 7 class logits. This is the *only* trainable component in
this notebook — everything upstream (MobileNetV3, CLIP) is frozen.

In [ ]:
class FusionHead(nn.Module):
    def __init__(self, image_dim=512, text_dim=512, hidden_dim=FUSION_HIDDEN_DIM, num_classes=NUM_CLASSES):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(image_dim + text_dim, hidden_dim),
            nn.Hardswish(),
            nn.Dropout(p=0.3),
            nn.Linear(hidden_dim, num_classes),
        )

    def forward(self, image_features, text_features):
        combined = torch.cat([image_features, text_features], dim=-1)
        return self.net(combined)

fusion_head = FusionHead(image_dim=512, text_dim=CLIP_EMBED_DIM).to(device)
print(f"FusionHead params (trainable, on-device at inference): {sum(p.numel() for p in fusion_head.parameters()):,}")

## Dataset
Pairs each real training image with a randomly-sampled proxy symptom-text template for its class, re-sampled every time an image is drawn (acts like text-side data augmentation, so the fusion head doesn't just memorize one exact string per class).

In [ ]:
class ImageTextDataset(Dataset):
    def __init__(self, csv_path, transform):
        self.df = pd.read_csv(csv_path)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        label = row['numeric_label']
        try:
            image = Image.open(row['image_path']).convert('RGB')
        except Exception:
            image = Image.new('RGB', (224, 224))
        image = self.transform(image)
        text = random.choice(SYMPTOM_TEXT_TEMPLATES[CLASS_NAMES[label]])
        return image, text, label


train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3, hue=0.1),
    transforms.RandomAffine(degrees=0, translate=(0.1, 0.1)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

eval_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

train_labels = pd.read_csv(TRAIN_CSV)['numeric_label'].values
class_counts = np.bincount(train_labels)
class_weights = 1.0 / class_counts
sample_weights = class_weights[train_labels]
sampler = WeightedRandomSampler(weights=sample_weights, num_samples=len(sample_weights), replacement=True)

train_dataset = ImageTextDataset(TRAIN_CSV, train_transform)
# num_workers=0: classes/datasets defined in a live notebook cell aren't reliably importable by
# spawned worker processes on native Windows (see 05_clip_distillation_blend.ipynb's note on this).
train_loader = DataLoader(train_dataset, batch_size=32, sampler=sampler, num_workers=0, pin_memory=True)

val_dataset = ImageTextDataset(VAL_CSV, eval_transform)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=0, pin_memory=True)

test_dataset = ImageTextDataset(TEST_CSV, eval_transform)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, num_workers=0, pin_memory=True)

print(f"Train batches: {len(train_loader)} | Val: {len(val_loader)} | Test: {len(test_loader)}")

## Training loop
Only `fusion_head.parameters()` are passed to the optimizer — MobileNetV3 and CLIP never receive a gradient update.

In [ ]:
optimizer = torch.optim.Adam(fusion_head.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', patience=3, factor=0.5)


def run_epoch(loader, train=True):
    fusion_head.train(train)
    running_loss = 0.0
    correct = 0
    total = 0

    for images, texts, labels in loader:
        images = images.to(device)
        labels = labels.to(device)

        image_features = extract_mobilenet_features(images)
        text_features = embed_symptom_text(list(texts))

        if train:
            optimizer.zero_grad()

        logits = fusion_head(image_features, text_features)
        loss = F.cross_entropy(logits, labels, label_smoothing=0.1)

        if train:
            loss.backward()
            optimizer.step()

        running_loss += loss.item()
        _, predicted = logits.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()

    return running_loss / len(loader), 100. * correct / total


best_val_loss = float('inf')
patience = 5
patience_counter = 0
history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}

for epoch in range(NUM_EPOCHS):
    train_loss, train_acc = run_epoch(train_loader, train=True)
    with torch.no_grad():
        val_loss, val_acc = run_epoch(val_loader, train=False)

    scheduler.step(val_loss)
    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        patience_counter = 0
        torch.save(fusion_head.state_dict(), FUSION_HEAD_CHECKPOINT)
        print(f"Epoch {epoch+1}/{NUM_EPOCHS} | Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.2f}% | Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.2f}% \u2713 saved")
    else:
        patience_counter += 1
        print(f"Epoch {epoch+1}/{NUM_EPOCHS} | Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.2f}% | Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.2f}% patience: {patience_counter}/{patience}")
        if patience_counter >= patience:
            print(f"\nEarly stopping triggered at epoch {epoch+1}")
            break

print("\nFusion head training complete!")

## Evaluation

Load the best fusion-head checkpoint and evaluate it on the test set, using one **fixed**
canonical description per class (index 0 of each template list) rather than a random one, so
this number is reproducible run to run — unlike training, eval shouldn't have a random
component.

In [ ]:
fusion_head.load_state_dict(torch.load(FUSION_HEAD_CHECKPOINT, map_location=device, weights_only=True))
fusion_head.eval()

canonical_texts_by_label = {i: SYMPTOM_TEXT_TEMPLATES[name][0] for i, name in enumerate(CLASS_NAMES)}

@torch.no_grad()
def evaluate_fused(loader, text_override=None):
    """text_override: None (use each example's own true-label canonical text) or a fixed
    string/list of strings to force onto every example (used for the mismatched-text check)."""
    all_preds, all_labels = [], []
    for images, texts, labels in loader:
        images = images.to(device)
        if text_override is None:
            eval_texts = [canonical_texts_by_label[int(l)] for l in labels]
        else:
            eval_texts = text_override if isinstance(text_override, list) else [text_override] * len(labels)

        image_features = extract_mobilenet_features(images)
        text_features = embed_symptom_text(eval_texts)
        logits = fusion_head(image_features, text_features)
        _, predicted = logits.max(1)

        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(labels.numpy())
    return all_labels, all_preds

fused_labels, fused_preds = evaluate_fused(test_loader)
print("── Fused (image + correct-class text) on the test set ──")
print(classification_report(fused_labels, fused_preds, target_names=CLASS_NAMES, zero_division=0))

## Does the model actually use the text? (mismatched-text sanity check)

If the fusion head learned to ignore the text and just classify off the image (i.e. the
learned weights on the text half of the concatenation collapsed to ~0), swapping in a random
wrong-class description would barely change accuracy. Feeding every test image the same,
deliberately unrelated description below and watching accuracy **drop** is the actual evidence
that the text branch carries real weight — not just wiring.

In [ ]:
wrong_text_labels, wrong_text_preds = evaluate_fused(test_loader, text_override="I have a broken arm from a bicycle accident.")
wrong_text_accuracy = accuracy_score(wrong_text_labels, wrong_text_preds)
correct_text_accuracy = accuracy_score(fused_labels, fused_preds)

print(f"Accuracy with correct-class text:   {correct_text_accuracy*100:.2f}%")
print(f"Accuracy with unrelated/wrong text: {wrong_text_accuracy*100:.2f}%")
print(f"Drop: {(correct_text_accuracy - wrong_text_accuracy)*100:.2f} points")
print()
if correct_text_accuracy - wrong_text_accuracy > 0.01:
    print("Text branch appears to carry real signal (accuracy drops with unrelated text).")
else:
    print("WARNING: little to no drop \u2014 the fusion head may be effectively ignoring the text.")

## Image-only baseline for comparison
Same frozen MobileNetV3, no text/fusion at all — the direct predecessor this branch is trying to improve on.

In [ ]:
@torch.no_grad()
def evaluate_image_only(loader):
    all_preds, all_labels = [], []
    for images, _texts, labels in loader:
        images = images.to(device)
        logits = student(images)
        _, predicted = logits.max(1)
        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(labels.numpy())
    return all_labels, all_preds

baseline_labels, baseline_preds = evaluate_image_only(test_loader)
print("── MobileNetV3 image-only baseline on the test set ──")
print(classification_report(baseline_labels, baseline_preds, target_names=CLASS_NAMES, zero_division=0))

## Benchmark table row
Formatted to drop into the shared benchmark table from `CONTEXT.md` — this is the "image+text accuracy" row for branch B.

In [ ]:
rows = [
    {"variant": "Baseline (image-only, no text, from 03_model_training.ipynb)",
     "on_device_params_at_inference": sum(p.numel() for p in student.parameters()),
     "test_accuracy": accuracy_score(baseline_labels, baseline_preds)},
    {"variant": "CLIP text-fusion (this notebook)",
     "on_device_params_at_inference": sum(p.numel() for p in student.parameters()) + sum(p.numel() for p in clip_model.parameters()) + sum(p.numel() for p in fusion_head.parameters()),
     "test_accuracy": accuracy_score(fused_labels, fused_preds)},
    {"variant": "CLIP text-fusion with unrelated text (sanity check, not a real deployment mode)",
     "on_device_params_at_inference": sum(p.numel() for p in student.parameters()) + sum(p.numel() for p in clip_model.parameters()) + sum(p.numel() for p in fusion_head.parameters()),
     "test_accuracy": accuracy_score(wrong_text_labels, wrong_text_preds)},
]

benchmark_df = pd.DataFrame(rows)
benchmark_df["test_accuracy"] = (benchmark_df["test_accuracy"] * 100).round(2)
print(benchmark_df.to_string(index=False))
print()
print("Unlike branches A and C, branch B's 'on-device' cost genuinely includes CLIP's text tower")
print("(151M params) plus the fusion head, since CLIP runs live in /predict_fused \u2014 this is the")
print("real efficiency-vs-correlation tradeoff to report, not a params-unchanged row like A/C's.")

## Summary for the defense

- **On-device cost, honestly stated**: this is the only branch where CLIP's parameter count
  (151M) is a real, permanent cost of the deployed path, not a training-time-only or
  online-optional one — `/predict_fused` calls `clip_model.encode_text` on every request.
  This is the direct trade this branch makes for literally correlating image and text.
- **Training data is synthetic, not real** — no (image, symptom-text, label) dataset exists in
  this repo. The proxy text templates are patient-phrased but hand-written, not sourced from
  real patients. This should be named as a limitation before a panelist finds it, and is the
  first thing worth re-testing if/when real symptom-text data becomes available.
- **The mismatched-text check is the concrete evidence the fusion is real**, not just "the code
  runs" — report both accuracy numbers, not just the fused one.
- **The multilingual pipeline (NLLB + the Roman-Urdu transliterator) lives in
  `inference/server.py`, not this notebook** — this notebook only proves the fusion *head*
  works in English; the translation stages are a separate, also-tested piece of the claim.